# 🧱 Part 02: BPE Tokenizer: Growing a Vocabulary From Statistics

> **Previous context**: Part 01 showed why raw text must become token IDs, and why character-level and word-level tokenization both have trade-offs.
> **Goal for this part**: Implement Byte Pair Encoding (BPE) by counting frequent neighboring pairs and merging them step by step.

Today we are solving one concrete confusion: what is the hidden mechanism behind this part of an LLM, and how can we rebuild it with small numbers before trusting a library?

## 0. Why BPE exists

BPE solves the tension between tiny character vocabularies and huge word vocabularies. It starts with small pieces and lets frequent patterns become larger tokens.

## 1. Core action

The whole algorithm repeats one move: count adjacent token pairs, find the most frequent pair, merge it into a new token, and record that merge rule.

## 2. Manual example

If ('l', 'o') appears most often, BPE creates 'lo'. Later ('lo', 'w') might become 'low'. The vocabulary is not magic; it grows from repeated statistics.

## 3. Industrial difference

Real tokenizers use byte-level handling, normalization rules, special tokens, and large corpora. This notebook keeps the algorithm small so every merge is visible.

## How to use the code cells

Run the cells in order. The code is intentionally direct and small: each cell should expose one idea, print the key observation, and let you change a number to see what moves.

## Exercises

When a cell contains a TODO placeholder, fill it yourself and use the `assert` checks as feedback. You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

## Summary Checklist

- [ ] BPE begins from small units and repeatedly merges frequent adjacent pairs.
- [ ] Merge rules define how encoding works later.
- [ ] Subword tokenization avoids most OOV failures while keeping sequences shorter than character-level tokenization.

Next, continue through the code cells for the Foundation part and inspect the printed observations.


In [ ]:
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog",
    "i love my cat",
    "i love my dog",
    "the cat is cute",
    "the dog is happy",
    "the mat is soft",
    "the log is hard",
    "cats and dogs are friends",
]

print('Read the values printed above and connect them to the concept in this cell.')
for i, s in enumerate(corpus):
    print(f"  [{i}] {s}")

In [ ]:
# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.

def text_to_tokens(text):
    'Read the values printed above and connect them to the concept in this cell.'
    return list(text)

# Teaching note: follow this line to see the main step.
sentence = corpus[0]
chars = text_to_tokens(sentence)

print('f"Original text:   \'{sentence}\'"')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

# Teaching note: follow this line to see the main step.
corpus_tokens = [text_to_tokens(s) for s in corpus]
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
from collections import defaultdict

def count_pairs(token_lists):
    'Read the values printed above and connect them to the concept in this cell.'
    pairs = defaultdict(int)
    for tokens in token_lists:
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            pairs[pair] += 1
    return dict(pairs)

# Teaching note: follow this line to see the main step.
initial_pairs = count_pairs(corpus_tokens)

print('Read the values printed above and connect them to the concept in this cell.')
# Teaching note: follow this line to see the main step.
for pair, count in sorted(initial_pairs.items(), key=lambda x: -x[1]):
    print('Read the values printed above and connect them to the concept in this cell.')

print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
def merge_pair(token_lists, pair_to_merge):
    'Read the values printed above and connect them to the concept in this cell.'
    a, b = pair_to_merge
    new_token = a + b  # Teaching note: follow this line to see the main step.
    
    new_token_lists = []
    for tokens in token_lists:
        new_tokens = []
        i = 0
        while i < len(tokens):
            # Teaching note: follow this line to see the main step.
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(new_token)
                i += 2  # Teaching note: follow this line to see the main step.
            else:
                new_tokens.append(tokens[i])
                i += 1
        new_token_lists.append(new_tokens)
    
    return new_token_lists, new_token

# Teaching note: follow this line to see the main step.
best_pair = max(initial_pairs, key=initial_pairs.get)
print('Read the values printed above and connect them to the concept in this cell.')

merged_tokens, new_token = merge_pair(corpus_tokens, best_pair)
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
for i, tokens in enumerate(merged_tokens):
    print(f"  [{i}] {tokens}")

In [ ]:
class BPETokenizer:
    'Read the values printed above and connect them to the concept in this cell.'
    
    def __init__(self):
        # Teaching note: follow this line to see the main step.
        # Teaching note: follow this line to see the main step.
        self.merge_rules = []
        # Teaching note: follow this line to see the main step.
        self.vocab = set()
        self.stoi = {}
        self.itos = {}
    
    def train(self, texts, num_merges=15, verbose=True):
        'Training'
        # Teaching note: follow this line to see the main step.
        token_lists = [list(text) for text in texts]
        base_vocab = set(c for tokens in token_lists for c in tokens)
        learned_vocab = set(base_vocab)
        
        if verbose:
            print(f"{'='*60}")
            print('Training')
            print(f"{'='*60}")
            print('Read the values printed above and connect them to the concept in this cell.')
            print()
        
        for step in range(num_merges):
            # Teaching note: follow this line to see the main step.
            pairs = defaultdict(int)
            for tokens in token_lists:
                for i in range(len(tokens) - 1):
                    pairs[(tokens[i], tokens[i + 1])] += 1
            
            if not pairs:
                break
            
            # Teaching note: follow this line to see the main step.
            best_pair = max(pairs, key=pairs.get)
            best_count = pairs[best_pair]
            
            # Teaching note: follow this line to see the main step.
            self.merge_rules.append(best_pair)
            
            # Teaching note: follow this line to see the main step.
            a, b = best_pair
            new_token = a + b
            
            new_token_lists = []
            for tokens in token_lists:
                new_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                        new_tokens.append(new_token)
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                new_token_lists.append(new_tokens)
            
            token_lists = new_token_lists
            
            learned_vocab.add(new_token)
            
            if verbose:
                print('Current vocabulary')
        
        # Teaching note: follow this line to see the main step.
        # Teaching note: follow this line to see the main step.
        # Teaching note: follow this line to see the main step.
        self.vocab = learned_vocab
        sorted_vocab = sorted(list(self.vocab))
        self.stoi = {t: i for i, t in enumerate(sorted_vocab)}
        self.itos = {i: t for i, t in enumerate(sorted_vocab)}
        
        if verbose:
            print(f"\n{'='*60}")
            print('f"Training finished！"')
            print('Read the values printed above and connect them to the concept in this cell.')
            print('Vocabulary')
            print(f"  - Merge rules: {self.merge_rules}")
            print(f"{'='*60}")
        
        return token_lists


In [ ]:
# Teaching note: follow this line to see the main step.
bpe = BPETokenizer()
final_tokens = bpe.train(corpus, num_merges=15, verbose=True)

In [ ]:
# Teaching note: follow this line to see the main step.
def bpe_encode(self, text):
    'Training'
    # Teaching note: follow this line to see the main step.
    tokens = list(text)
    
    # Teaching note: follow this line to see the main step.
    for (a, b) in self.merge_rules:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(a + b)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    
    # Teaching note: follow this line to see the main step.
    ids = []
    for token in tokens:
        if token in self.stoi:
            ids.append(self.stoi[token])
        else:
            # Teaching note: follow this line to see the main step.
            for ch in token:
                ids.append(self.stoi[ch])
    return ids

# Teaching note: follow this line to see the main step.
BPETokenizer.encode = bpe_encode

print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
text = "the cat"
ids = bpe.encode(text)
print('f"Original text: \'{text}\'"')
print(f"Token IDs: {ids}")
print('f"Number of tokens: {len(ids)}"')

# Teaching note: follow this line to see the main step.
print('Explanation: the printed values show the main mechanism in this step.')
for i, tid in enumerate(ids):
    print(f"  ID {tid} → '{bpe.itos[tid]}'")

# Teaching note: follow this line to see the main step.
print('Number of tokens')

In [ ]:
def bpe_decode(self, ids):
    'Decoded text'
    return "".join([self.itos[i] for i in ids])

BPETokenizer.decode = bpe_decode

# Teaching note: follow this line to see the main step.
for test_text in ["the cat", "my dog is happy", "i love cats"]:
    ids = bpe.encode(test_text)
    recovered = bpe.decode(ids)
    status = "[ok]" if test_text == recovered else "[x]"
    print(f"{status} '{test_text}' → {ids} → '{recovered}'")

In [ ]:
# Teaching note: follow this line to see the main step.
unseen = "a cat runs fast"
print('Read the values printed above and connect them to the concept in this cell.')
print('Training')

ids = bpe.encode(unseen)
recovered = bpe.decode(ids)

print(f"\nToken IDs: {ids}")
print('Decoded text')
print('Read the values printed above and connect them to the concept in this cell.')

# Teaching note: follow this line to see the main step.
print('Inspect each token:')
for tid in ids:
    print(f"  '{bpe.itos[tid]}'", end="")
print()

In [ ]:
# Teaching note: follow this line to see the main step.
for n_merges in [5, 15, 30]:
    t = BPETokenizer()
    t.train(corpus, num_merges=n_merges, verbose=False)
    
    test = "the cat sat on the mat"
    ids = t.encode(test)
    
    print('Vocabulary')

print()
print('Number of tokens')

In [ ]:
# Teaching note: follow this line to see the main step.
try:
    import tiktoken

    real_tokenizer_name = "gpt2"
    real_tokenizer = tiktoken.get_encoding(real_tokenizer_name)
    print('Real tokenizer: GPT-2 byte-level BPE')
    print('f"Vocabulary size: {real_tokenizer.n_vocab}"')
except Exception as e:
    real_tokenizer = None
    print('Could not load tiktoken. Install tiktoken to run the real tokenizer demo.')
    print('Reason')
    print('Real tokenizer: GPT-2 byte-level BPE')


def show_real_tokenization(text):
    'Real tokenizer: GPT-2 byte-level BPE'
    ids = real_tokenizer.encode(text)
    tokens = [real_tokenizer.decode([tok_id]) for tok_id in ids]

    print('Read the values printed above and connect them to the concept in this cell.')
    print('Number of tokens')
    for i, (tok_id, token) in enumerate(zip(ids, tokens)):
        visible = token.replace("\n", "\\n").replace("\t", "\\t")
        token_bytes = list(real_tokenizer.decode_single_token_bytes(tok_id))
        print(f"  {i:02d} | id={tok_id:<6} | token={visible!r} | bytes={token_bytes}")
    return ids, tokens


if real_tokenizer is not None:
    show_real_tokenization("the cat sat on the mat")


In [ ]:
def try_mini_bpe(text):
    'Reason'
    try:
        ids = bpe.encode(text)
        tokens = [bpe.itos[i] for i in ids]
        return ids, tokens, None
    except Exception as e:
        return None, None, e


compare_texts = [
    "the cat sat on the mat",
    " the cat",
    "the  cat",
    "327 + 1 = 328",
    "def f():\n    return x",
    'Read the values printed above and connect them to the concept in this cell.',
]

if real_tokenizer is not None:
    for text in compare_texts:
        print("=" * 68)
        print('Read the values printed above and connect them to the concept in this cell.')

        mini_ids, mini_tokens, error = try_mini_bpe(text)
        if error is None:
            print(f"mini BPE: {len(mini_ids)} tokens")
            print(f"  tokens: {mini_tokens}")
            print(f"  ids:    {mini_ids}")
        else:
            print('Read the values printed above and connect them to the concept in this cell.')
            print('Reason')

        real_ids = real_tokenizer.encode(text)
        real_tokens = [real_tokenizer.decode([tok_id]) for tok_id in real_ids]
        real_tokens = [t.replace("\n", "\\n") for t in real_tokens]
        print('Read the values printed above and connect them to the concept in this cell.')
        print(f"  tokens: {real_tokens}")
        print(f"  ids:    {real_ids}")

    print("=" * 68)
    print('Real tokenizer: GPT-2 byte-level BPE')
    print('Read the values printed above and connect them to the concept in this cell.')


In [ ]:
special_cases = [
    ('Read the values printed above and connect them to the concept in this cell.', "327 + 1 = 328"),
    ('Read the values printed above and connect them to the concept in this cell.', "cat"),
    ('Read the values printed above and connect them to the concept in this cell.', " cat"),
    ('Read the values printed above and connect them to the concept in this cell.', "the    cat"),
    ('Read the values printed above and connect them to the concept in this cell.', "def f():\n    return x"),
    ('Read the values printed above and connect them to the concept in this cell.', 'Read the values printed above and connect them to the concept in this cell.'),
]

if real_tokenizer is not None:
    for label, text in special_cases:
        print("=" * 68)
        print('Read the values printed above and connect them to the concept in this cell.')
        show_real_tokenization(text)

    print("=" * 68)
    print("Special token: <|endoftext|>")
    text = "hello<|endoftext|>world"

    try:
        real_tokenizer.encode(text)
    except ValueError as e:
        print('Read the values printed above and connect them to the concept in this cell.')
        print('Read the values printed above and connect them to the concept in this cell.')

    ids = real_tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    tokens = [real_tokenizer.decode([tok_id]) for tok_id in ids]
    print('Read the values printed above and connect them to the concept in this cell.')
    print(f"  ids:    {ids}")
    print(f"  tokens: {tokens}")
    print('Read the values printed above and connect them to the concept in this cell.')


In [ ]:
# Teaching note: follow this line to see the main step.
number_texts = ["1.11", "1.9"]

if real_tokenizer is not None:
    for text in number_texts:
        ids = real_tokenizer.encode(text)
        tokens = [real_tokenizer.decode([tok_id]) for tok_id in ids]
        print(f"{text!r} -> tokens={tokens}, ids={ids}")

    print()
    print('Read the values printed above and connect them to the concept in this cell.')
    print(f"  1.11 > 1.9  ? {1.11 > 1.9}")
    print(f"  1.90 > 1.11 ? {1.90 > 1.11}")
    print()
    print('Key observation: inspect the values above and connect them to the idea in this cell.')
    print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
print('Training')
print('Read the values printed above and connect them to the concept in this cell.')
print()

for i, (a, b) in enumerate(bpe.merge_rules):
    arrow = ""
    # Teaching note: follow this line to see the main step.
    merged = a + b
    if merged in ['th', 'the ', 'the c', 'the cat ']:
        arrow = 'Read the values printed above and connect them to the concept in this cell.'
    if a in ['th', 'the ', 'the c'] or b in ['e ', 'c', 'at ']:
        arrow = 'Read the values printed above and connect them to the concept in this cell.'
    print(f"  Rule {i+1:2d}: '{a}' + '{b}' → '{merged}'{arrow}")

print()
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
from collections import defaultdict

tokens = list("banana")
pair_counts = defaultdict(int)

# Teaching note: follow this line to see the main step.
'TODO: replace this placeholder with your code'

assert pair_counts[("a", "n")] == 2, dict(pair_counts)
assert pair_counts[("n", "a")] == 2, dict(pair_counts)
assert pair_counts[("b", "a")] == 1, dict(pair_counts)
print('Exercise passed: you have understood this step.')

In [ ]:
# Teaching note: follow this line to see the main step.
def merge_pair(tokens, pair):
    'Read the values printed above and connect them to the concept in this cell.'
    merged = []
    i = 0
    while i < len(tokens):
        # Teaching note: follow this line to see the main step.
        'TODO: replace this placeholder with your code'
    return merged

result = merge_pair(list("banana"), ("a", "n"))

assert result == ["b", "an", "an", "a"], result
print('Exercise passed: you have understood this step.')

In [ ]:
# Teaching note: follow this line to see the main step.
text = '"you😊"'

# Teaching note: follow this line to see the main step.
byte_values = 'TODO: replace this placeholder with your code'

assert not isinstance(byte_values, str), 'Please replace the placeholder before running the assertion.'
assert byte_values == [228, 189, 160, 240, 159, 152, 138], byte_values
print('Exercise passed: you have understood this step.')